In [ ]:
from pathlib import Path
import csv, random, time
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data_dir=Path('mnist_data'); output_dir=Path('outputs'); output_dir.mkdir(exist_ok=True)
train_data=datasets.MNIST(data_dir,train=True,download=True,transform=transforms.ToTensor())
test_data=datasets.MNIST(data_dir,train=False,download=True,transform=transforms.ToTensor())
generator=torch.Generator().manual_seed(SEED)
train_loader=torch.utils.data.DataLoader(train_data,batch_size=128,shuffle=True,generator=generator,num_workers=2,pin_memory=torch.cuda.is_available())
test_loader=torch.utils.data.DataLoader(test_data,batch_size=512,shuffle=False,num_workers=2,pin_memory=torch.cuda.is_available())
print('device:',device)

In [ ]:
MEAN,STD=0.1307,0.3081
class Net(nn.Module):
    def __init__(self):
        super().__init__(); self.fc1=nn.Linear(784,200); self.fc2=nn.Linear(200,10)
    def forward(self,x):
        x=((x-MEAN)/STD).flatten(1)
        return self.fc2(F.relu(self.fc1(x)))
model=Net().to(device)
print(model)

In [ ]:
def train_model(model,epochs=15):
    opt=torch.optim.Adam(model.parameters(),lr=1e-3); loss_fn=nn.CrossEntropyLoss()
    for epoch in range(1,epochs+1):
        model.train(); loss_sum=0.; correct=total=0
        for x,y in train_loader:
            x,y=x.to(device),y.to(device); opt.zero_grad(set_to_none=True); logits=model(x)
            loss=loss_fn(logits,y); loss.backward(); opt.step()
            loss_sum+=loss.item()*y.numel(); correct+=(logits.argmax(1)==y).sum().item(); total+=y.numel()
        print(f'Epoch {epoch:2d}/15  loss={loss_sum/total:.4f}  train_acc={correct/total:.2%}')
@torch.no_grad()
def clean_accuracy(model):
    model.eval(); correct=total=0
    for x,y in test_loader:
        x,y=x.to(device),y.to(device); correct+=(model(x).argmax(1)==y).sum().item(); total+=y.numel()
    return correct/total
checkpoint=output_dir/'mnist_interval_model.pt'; RETRAIN=False
if checkpoint.exists() and not RETRAIN:
    model.load_state_dict(torch.load(checkpoint,map_location=device,weights_only=True)); print('Loaded',checkpoint)
else:
    train_model(model,15); torch.save(model.state_dict(),checkpoint)
clean_acc=clean_accuracy(model)
print(f'Clean test accuracy: {clean_acc:.2%}')

In [ ]:
def interval_linear(lower,upper,weight,bias=None):
    wp,wn=weight.clamp(min=0),weight.clamp(max=0)
    if weight.ndim==2:
        lo=lower@wp.T+upper@wn.T; hi=upper@wp.T+lower@wn.T
    elif weight.ndim==3:
        lo=(wp*lower[:,None,:]+wn*upper[:,None,:]).sum(-1)
        hi=(wp*upper[:,None,:]+wn*lower[:,None,:]).sum(-1)
    else: raise ValueError('weight must have rank 2 or 3')
    return (lo,hi) if bias is None else (lo+bias,hi+bias)
@torch.no_grad()
def margin_lower_bounds(model,images,labels,eps):
    lo=((images-eps).clamp(0,1)-MEAN)/STD; hi=((images+eps).clamp(0,1)-MEAN)/STD
    lo,hi=interval_linear(lo.flatten(1),hi.flatten(1),model.fc1.weight,model.fc1.bias)
    lo,hi=lo.clamp(min=0),hi.clamp(min=0)
    wd=model.fc2.weight[labels,None,:]-model.fc2.weight[None,:,:]
    bd=model.fc2.bias[labels,None]-model.fc2.bias[None,:]
    return interval_linear(lo,hi,wd,bd)[0]
model.eval(); sx,sy=next(iter(test_loader)); sx,sy=sx[:8].to(device),sy[:8].to(device)
lb=margin_lower_bounds(model,sx,sy,.05)
for _ in range(20):
    px=(sx+torch.empty_like(sx).uniform_(-.05,.05)).clamp(0,1); logits=model(px)
    concrete=logits.gather(1,sy[:,None])-logits
    assert torch.all(concrete+1e-5>=lb)
print('Sampled soundness sanity check passed.')

In [ ]:
@torch.no_grad()
def verified_accuracy(model,eps):
    model.eval(); verified=total=0; start=time.perf_counter()
    for x,y in test_loader:
        x,y=x.to(device),y.to(device); lower=margin_lower_bounds(model,x,y,eps)
        lower.scatter_(1,y[:,None],float('inf'))
        verified+=(lower.min(1).values>=0).sum().item(); total+=y.numel()
    return verified/total,time.perf_counter()-start
epsilons=np.linspace(.01,.10,10); results=[]
print(f'Clean accuracy: {clean_acc:.2%}'); print(f"{'epsilon':>8} {'verified':>10} {'seconds':>10}")
for eps in epsilons:
    acc,seconds=verified_accuracy(model,float(eps)); results.append((float(eps),acc,seconds))
    print(f'{eps:8.2f} {acc:10.2%} {seconds:10.2f}')
with (output_dir/'interval_results.csv').open('w',newline='') as f:
    w=csv.writer(f); w.writerow(['epsilon','verified_accuracy','seconds']); w.writerows(results)

In [ ]:
plt.figure(figsize=(6.5,4)); plt.plot([r[0] for r in results],[100*r[1] for r in results],marker='o')
plt.xlabel(r'$\ell_\infty$ radius $\epsilon$'); plt.ylabel('Verified accuracy (%)')
plt.title('MNIST interval-analysis certificate'); plt.grid(alpha=.3); plt.ylim(bottom=0); plt.tight_layout()
plt.savefig(output_dir/'verified_accuracy.png',dpi=180); plt.show()
print('Artifacts saved under',output_dir.resolve())